In [20]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import warnings
 
warnings.filterwarnings("ignore")
matplotlib.use("Agg") 

# Colour palette

In [21]:
BG    = "#0F172A"
CARD  = "#1E293B"
TXT   = "#F1F5F9"
GRID  = "#334155"
ACC1  = "#3B82F6"   # blue  – No Disease
ACC2  = "#EF4444"   # red   – Disease
ACC3  = "#10B981"   # green
ACC4  = "#F59E0B"   # amber
ACC5  = "#A78BFA"   # violet
 
PALETTE = {0: ACC1, 1: ACC2}
 
OUTPUT_PREFIX = "eda_step"   # saved PNGs will be: eda_step1.png, eda_step2.png …
 
 
def savefig(fig, step: int, label: str):
    fname = f"{OUTPUT_PREFIX}{step:02d}_{label}.png"
    fig.savefig(fname, dpi=130, bbox_inches="tight", facecolor=BG)
    plt.close(fig)
    print(f"  ✔  Saved → {fname}")
 
 
def make_fig(rows, cols, title, h_per_row=5, w_per_col=7):
    fig = plt.figure(figsize=(cols * w_per_col, rows * h_per_row), facecolor=BG)
    fig.suptitle(title, fontsize=16, color=TXT, fontweight="bold", y=0.99)
    return fig
 
 
def style_ax(ax, title=""):
    ax.set_facecolor(CARD)
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.tick_params(colors=TXT, labelsize=9)
    ax.xaxis.label.set_color(TXT)
    ax.yaxis.label.set_color(TXT)
    if title:
        ax.set_title(title, color=TXT, fontsize=11, fontweight="bold", pad=9)
    return ax


# LOAD DATASET

In [22]:
print("\n" + "═" * 65)
print("  STEP 1 — Load Dataset & First Look")
print("═" * 65)
 
RAW_PATH = "hd.csv"   # ← update path if needed
df_raw = pd.read_csv(RAW_PATH)
 
print(f"\n  Dataset loaded successfully!")
print(f"  Rows : {df_raw.shape[0]:,}")
print(f"  Cols : {df_raw.shape[1]}")
print(f"\n  First 5 rows:")
print(df_raw.head().to_string())
print(f"\n  Last 5 rows:")
print(df_raw.tail().to_string())


═════════════════════════════════════════════════════════════════
  STEP 1 — Load Dataset & First Look
═════════════════════════════════════════════════════════════════

  Dataset loaded successfully!
  Rows : 920
  Cols : 16

  First 5 rows:
   id  age     sex    dataset               cp  trestbps   chol    fbs         restecg  thalch  exang  oldpeak        slope   ca               thal  num
0   1   63    Male  Cleveland   typical angina     145.0  233.0   True  lv hypertrophy   150.0  False      2.3  downsloping  0.0       fixed defect    0
1   2   67    Male  Cleveland     asymptomatic     160.0  286.0  False  lv hypertrophy   108.0   True      1.5         flat  3.0             normal    2
2   3   67    Male  Cleveland     asymptomatic     120.0  229.0  False  lv hypertrophy   129.0   True      2.6         flat  2.0  reversable defect    1
3   4   37    Male  Cleveland      non-anginal     130.0  250.0  False          normal   187.0  False      3.5  downsloping  0.0             nor

# SHAPE, COLUMNS & DATA TYPES

In [23]:
print("\n" + "═" * 65)
print("  STEP 2 — Shape, Columns & Data Types")
print("═" * 65)
 
print(f"\n  Shape  : {df_raw.shape}")
print(f"\n  Columns ({len(df_raw.columns)}):")
for col in df_raw.columns:
    print(f"    • {col:<12}  dtype={df_raw[col].dtype}  "
          f"unique={df_raw[col].nunique()}")
 
# Visual: dtype breakdown bar chart
fig = make_fig(1, 1, "Step 2 — Column Data Types & Unique Value Counts", h_per_row=5, w_per_col=14)
ax = fig.add_subplot(111)
style_ax(ax, "Unique Value Count per Column")
cols_sorted = df_raw.nunique().sort_values(ascending=False)
colors_dtype = []
for col in cols_sorted.index:
    dt = str(df_raw[col].dtype)
    if "int" in dt:   colors_dtype.append(ACC1)
    elif "float" in dt: colors_dtype.append(ACC3)
    else:             colors_dtype.append(ACC4)
 
bars = ax.bar(cols_sorted.index, cols_sorted.values, color=colors_dtype, edgecolor="none", width=0.6)
for b, v in zip(bars, cols_sorted.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.5, str(v),
            ha="center", color=TXT, fontsize=9, fontweight="bold")
ax.set_ylabel("Unique Values")
ax.tick_params(axis="x", rotation=35)
ax.yaxis.grid(True, color=GRID, lw=0.5); ax.set_axisbelow(True)
 
# legend
from matplotlib.patches import Patch
legend_els = [Patch(facecolor=ACC1, label="int"), Patch(facecolor=ACC3, label="float"),
              Patch(facecolor=ACC4, label="object/str")]
ax.legend(handles=legend_els, frameon=False, labelcolor=TXT, fontsize=9)
savefig(fig, 2, "dtypes_unique_counts")


═════════════════════════════════════════════════════════════════
  STEP 2 — Shape, Columns & Data Types
═════════════════════════════════════════════════════════════════

  Shape  : (920, 16)

  Columns (16):
    • id            dtype=int64  unique=920
    • age           dtype=int64  unique=50
    • sex           dtype=str  unique=2
    • dataset       dtype=str  unique=4
    • cp            dtype=str  unique=4
    • trestbps      dtype=float64  unique=61
    • chol          dtype=float64  unique=217
    • fbs           dtype=object  unique=2
    • restecg       dtype=str  unique=3
    • thalch        dtype=float64  unique=119
    • exang         dtype=object  unique=2
    • oldpeak       dtype=float64  unique=53
    • slope         dtype=str  unique=3
    • ca            dtype=float64  unique=4
    • thal          dtype=str  unique=3
    • num           dtype=int64  unique=5
  ✔  Saved → eda_step02_dtypes_unique_counts.png


# MISSING VALUE ANALYSIS

In [24]:
print("\n" + "═" * 65)
print("  STEP 3 — Missing Value Analysis")
print("═" * 65)
 
miss = df_raw.isnull().sum()
miss_pct = (miss / len(df_raw) * 100).round(2)
miss_df = pd.DataFrame({"Missing Count": miss, "Missing %": miss_pct})
miss_df = miss_df[miss_df["Missing Count"] > 0].sort_values("Missing %", ascending=False)
 
print(f"\n  Columns with missing values:")
print(miss_df.to_string())
print(f"\n  Total cells     : {df_raw.shape[0] * df_raw.shape[1]:,}")
print(f"  Total missing   : {df_raw.isnull().sum().sum():,}")
print(f"  Overall missing%: {df_raw.isnull().sum().sum() / (df_raw.shape[0]*df_raw.shape[1])*100:.2f}%")
 
# Visual
fig = make_fig(1, 2, "Step 3 — Missing Value Analysis", h_per_row=5, w_per_col=8)
gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)
 
ax1 = fig.add_subplot(gs[0])
style_ax(ax1, "Missing Count per Column")
bar_c = [ACC2 if v > 300 else ACC4 if v > 50 else ACC3 for v in miss_df["Missing Count"]]
ax1.barh(miss_df.index, miss_df["Missing Count"], color=bar_c, edgecolor="none", height=0.6)
for i, (idx_v, val) in enumerate(zip(miss_df.index, miss_df["Missing Count"])):
    ax1.text(val + 5, i, f"{val} ({miss_df.loc[idx_v,'Missing %']}%)", va="center", color=TXT, fontsize=9)
ax1.set_xlabel("Missing Count")
ax1.xaxis.grid(True, color=GRID, lw=0.5); ax1.set_axisbelow(True)
 
ax2 = fig.add_subplot(gs[1])
style_ax(ax2, "Missing % Heatmap (column-wise)")
miss_matrix = df_raw.isnull().astype(int)
col_order = miss_matrix.sum().sort_values(ascending=False).index
miss_matrix = miss_matrix[col_order]
sns.heatmap(miss_matrix.T, ax=ax2, cmap="Reds", cbar=False,
            xticklabels=False, yticklabels=True)
ax2.tick_params(colors=TXT, labelsize=8)
ax2.set_xlabel("Samples (rows)", color=TXT); ax2.set_ylabel("")
savefig(fig, 3, "missing_values")



═════════════════════════════════════════════════════════════════
  STEP 3 — Missing Value Analysis
═════════════════════════════════════════════════════════════════

  Columns with missing values:
          Missing Count  Missing %
ca                  611      66.41
thal                486      52.83
slope               309      33.59
fbs                  90       9.78
oldpeak              62       6.74
trestbps             59       6.41
exang                55       5.98
thalch               55       5.98
chol                 30       3.26
restecg               2       0.22

  Total cells     : 14,720
  Total missing   : 1,759
  Overall missing%: 11.95%
  ✔  Saved → eda_step03_missing_values.png


# DATA CLEANING & IMPUTATION

In [25]:
print("\n" + "═" * 65)
print("  STEP 4 — Data Cleaning & Imputation")
print("═" * 65)
 
df = df_raw.drop(columns=["id", "dataset"]).copy()
 
# Binary target
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns=["num"])
 
# Separate column types
str_cols = [c for c in df.columns if df[c].dtype == object or str(df[c].dtype) == "str"]
num_cols = [c for c in df.columns if c not in str_cols and c != "target"]
 
print(f"\n  Categorical columns : {str_cols}")
print(f"  Numeric columns     : {num_cols}")
print(f"\n  Imputation strategy:")
print(f"    • Categorical → mode (most frequent value)")
print(f"    • Numeric     → median (robust to outliers)")
 
before_miss = df.isnull().sum().sum()
 
for col in str_cols:
    mode_val = df[col].dropna().mode()[0]
    df[col] = df[col].fillna(mode_val)
    print(f"    ✔ {col:<12} filled with mode='{mode_val}'")
 
for col in num_cols:
    med_val = df[col].dropna().median()
    df[col] = df[col].fillna(med_val)
    print(f"    ✔ {col:<12} filled with median={med_val:.2f}")
 
after_miss = df.isnull().sum().sum()
print(f"\n  Missing before: {before_miss}  →  after: {after_miss}  ✅")
 
# Encode for correlation later
df_enc = df.copy()
le = LabelEncoder()
for col in str_cols:
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))



═════════════════════════════════════════════════════════════════
  STEP 4 — Data Cleaning & Imputation
═════════════════════════════════════════════════════════════════

  Categorical columns : ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']
  Numeric columns     : ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca']

  Imputation strategy:
    • Categorical → mode (most frequent value)
    • Numeric     → median (robust to outliers)
    ✔ sex          filled with mode='Male'
    ✔ cp           filled with mode='asymptomatic'
    ✔ fbs          filled with mode='False'
    ✔ restecg      filled with mode='normal'
    ✔ exang        filled with mode='False'
    ✔ slope        filled with mode='flat'
    ✔ thal         filled with mode='normal'
    ✔ age          filled with median=54.00
    ✔ trestbps     filled with median=130.00
    ✔ chol         filled with median=223.00
    ✔ thalch       filled with median=140.00
    ✔ oldpeak      filled with median=0.50
    ✔ ca   

# TARGET VARIABLE ANALYSIS

In [26]:
print("\n" + "═" * 65)
print("  STEP 5 — Target Variable Analysis")
print("═" * 65)
 
target_counts = df["target"].value_counts().sort_index()
print(f"\n  Class distribution:")
print(f"    No Disease (0) : {target_counts[0]:>4}  ({target_counts[0]/len(df)*100:.1f}%)")
print(f"    Disease    (1) : {target_counts[1]:>4}  ({target_counts[1]/len(df)*100:.1f}%)")
print(f"    Class imbalance ratio: {target_counts[1]/target_counts[0]:.2f}")
 
fig = make_fig(1, 2, "Step 5 — Target Variable Distribution", h_per_row=5, w_per_col=6)
gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.4)
 
ax1 = fig.add_subplot(gs[0])
style_ax(ax1, "Count")
bars = ax1.bar(["No Disease", "Disease"], target_counts.values, color=[ACC1, ACC2], width=0.5)
for b, v in zip(bars, target_counts.values):
    ax1.text(b.get_x() + b.get_width() / 2, v + 5,
             f"{v}\n({v/len(df)*100:.1f}%)", ha="center", color=TXT, fontsize=10, fontweight="bold")
ax1.set_ylabel("Count"); ax1.set_ylim(0, max(target_counts) * 1.2)
ax1.yaxis.grid(True, color=GRID, lw=0.5); ax1.set_axisbelow(True)
 
ax2 = fig.add_subplot(gs[1])
style_ax(ax2, "Proportion")
wedges, texts, autotexts = ax2.pie(
    target_counts, labels=["No Disease", "Disease"],
    colors=[ACC1, ACC2], autopct="%1.1f%%", startangle=90,
    wedgeprops=dict(edgecolor=BG, linewidth=2.5))
for t in texts: t.set_color(TXT); t.set_fontsize(10)
for at in autotexts: at.set_color(TXT); at.set_fontweight("bold")
savefig(fig, 5, "target_distribution")



═════════════════════════════════════════════════════════════════
  STEP 5 — Target Variable Analysis
═════════════════════════════════════════════════════════════════

  Class distribution:
    No Disease (0) :  411  (44.7%)
    Disease    (1) :  509  (55.3%)
    Class imbalance ratio: 1.24
  ✔  Saved → eda_step05_target_distribution.png


In [27]:
print("\n" + "═" * 65)
print("  STEP 6 — Univariate Analysis: Numeric Features")
print("═" * 65)
 
print(f"\n  Descriptive statistics:")
print(df[num_cols].describe().round(2).to_string())
 
fig = make_fig(2, 3, "Step 6 — Univariate: Numeric Features (Histogram + KDE)", h_per_row=4, w_per_col=7)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
 
for i, col in enumerate(num_cols):
    ax = fig.add_subplot(gs[i // 3, i % 3])
    style_ax(ax, col)
    data = df[col].dropna()
    ax.hist(data, bins=25, color=ACC1, alpha=0.6, edgecolor="none", density=True, label="histogram")
    data.plot.kde(ax=ax, color=ACC4, lw=2, label="KDE")
    ax.axvline(data.mean(), color=ACC2, lw=1.5, ls="--", label=f"mean={data.mean():.1f}")
    ax.axvline(data.median(), color=ACC3, lw=1.5, ls=":", label=f"median={data.median():.1f}")
    ax.set_ylabel("Density")
    ax.legend(frameon=False, labelcolor=TXT, fontsize=7.5)
    ax.yaxis.grid(True, color=GRID, lw=0.5); ax.set_axisbelow(True)
    skew = data.skew()
    ax.text(0.97, 0.96, f"skew={skew:.2f}", transform=ax.transAxes,
            ha="right", va="top", color=ACC4, fontsize=8)
    print(f"    {col:<12}  mean={data.mean():.2f}  median={data.median():.2f}  "
          f"std={data.std():.2f}  skew={skew:.2f}")
 
savefig(fig, 6, "univariate_numeric")



═════════════════════════════════════════════════════════════════
  STEP 6 — Univariate Analysis: Numeric Features
═════════════════════════════════════════════════════════════════

  Descriptive statistics:
          age  trestbps    chol  thalch  oldpeak      ca
count  920.00    920.00  920.00  920.00   920.00  920.00
mean    53.51    132.00  199.91  137.69     0.85    0.23
std      9.42     18.45  109.04   25.15     1.06    0.63
min     28.00      0.00    0.00   60.00    -2.60    0.00
25%     47.00    120.00  177.75  120.00     0.00    0.00
50%     54.00    130.00  223.00  140.00     0.50    0.00
75%     60.00    140.00  267.00  156.00     1.50    0.00
max     77.00    200.00  603.00  202.00     6.20    3.00
    age           mean=53.51  median=54.00  std=9.42  skew=-0.20
    trestbps      mean=132.00  median=130.00  std=18.45  skew=0.24
    chol          mean=199.91  median=223.00  std=109.04  skew=-0.64
    thalch        mean=137.69  median=140.00  std=25.15  skew=-0.24
    oldpe

In [28]:
print("\n" + "═" * 65)
print("  STEP 7 — Univariate Analysis: Categorical Features")
print("═" * 65)
 
n_cat = len(str_cols)
n_cat_cols = 3
n_cat_rows = (n_cat + n_cat_cols - 1) // n_cat_cols
fig = make_fig(n_cat_rows, n_cat_cols, "Step 7 — Univariate: Categorical Features", h_per_row=4, w_per_col=7)
gs = gridspec.GridSpec(n_cat_rows, n_cat_cols, figure=fig, hspace=0.50, wspace=0.35)
 
for i, col in enumerate(str_cols):
    ax = fig.add_subplot(gs[i // 3, i % 3])
    style_ax(ax, col)
    vc = df[col].value_counts()
    print(f"\n    {col}:")
    print(f"    {vc.to_dict()}")
    colors_cat = [ACC1, ACC2, ACC3, ACC4, ACC5][:len(vc)]
    bars = ax.bar(vc.index, vc.values, color=colors_cat, edgecolor="none", width=0.6)
    for b, v in zip(bars, vc.values):
        ax.text(b.get_x() + b.get_width() / 2, v + 1,
                f"{v}\n({v/len(df)*100:.0f}%)", ha="center", color=TXT, fontsize=8)
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=20)
    ax.yaxis.grid(True, color=GRID, lw=0.5); ax.set_axisbelow(True)
 
savefig(fig, 7, "univariate_categorical")



═════════════════════════════════════════════════════════════════
  STEP 7 — Univariate Analysis: Categorical Features
═════════════════════════════════════════════════════════════════

    sex:
    {'Male': 726, 'Female': 194}

    cp:
    {'asymptomatic': 496, 'non-anginal': 204, 'atypical angina': 174, 'typical angina': 46}

    fbs:
    {False: 782, True: 138}

    restecg:
    {'normal': 553, 'lv hypertrophy': 188, 'st-t abnormality': 179}

    exang:
    {False: 583, True: 337}

    slope:
    {'flat': 654, 'upsloping': 203, 'downsloping': 63}

    thal:
    {'normal': 682, 'reversable defect': 192, 'fixed defect': 46}
  ✔  Saved → eda_step07_univariate_categorical.png


In [29]:
print("\n" + "═" * 65)
print("  STEP 8 — Bivariate Analysis: Numeric Features vs Target")
print("═" * 65)
 
fig = make_fig(2, 3, "Step 8 — Bivariate: Numeric Features vs Target (Box + Strip)", h_per_row=4, w_per_col=7)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.50, wspace=0.35)
 
for i, col in enumerate(num_cols):
    ax = fig.add_subplot(gs[i // 3, i % 3])
    style_ax(ax, col)
    groups = [df[df["target"] == t][col].dropna() for t in [0, 1]]
    bp = ax.boxplot(groups, patch_artist=True, widths=0.45,
                    medianprops=dict(color=TXT, lw=2.2),
                    whiskerprops=dict(color=TXT), capprops=dict(color=TXT),
                    flierprops=dict(marker="o", color=ACC4, markersize=2.5, alpha=0.5))
    for patch, col_b in zip(bp["boxes"], [ACC1, ACC2]):
        patch.set_facecolor(col_b); patch.set_alpha(0.75)
    # overlay strip
    for j, (grp, c) in enumerate(zip(groups, [ACC1, ACC2])):
        jitter = np.random.uniform(-0.12, 0.12, size=len(grp))
        ax.scatter(np.full(len(grp), j + 1) + jitter, grp,
                   alpha=0.18, s=6, color=c, edgecolors="none")
    ax.set_xticks([1, 2]); ax.set_xticklabels(["No Disease", "Disease"])
    ax.set_ylabel(col)
    ax.yaxis.grid(True, color=GRID, lw=0.5); ax.set_axisbelow(True)
    mean_diff = groups[1].mean() - groups[0].mean()
    direction = "↑" if mean_diff > 0 else "↓"
    ax.text(0.97, 0.97, f"Δmean={mean_diff:+.1f} {direction}", transform=ax.transAxes,
            ha="right", va="top", color=ACC4, fontsize=8)
    print(f"    {col:<12}  mean(no_dis)={groups[0].mean():.2f}  "
          f"mean(disease)={groups[1].mean():.2f}  Δ={mean_diff:+.2f}")
 
savefig(fig, 8, "bivariate_numeric_vs_target")



═════════════════════════════════════════════════════════════════
  STEP 8 — Bivariate Analysis: Numeric Features vs Target
═════════════════════════════════════════════════════════════════
    age           mean(no_dis)=50.55  mean(disease)=55.90  Δ=+5.36
    trestbps      mean(no_dis)=129.92  mean(disease)=133.67  Δ=+3.76
    chol          mean(no_dis)=227.68  mean(disease)=177.49  Δ=-50.19
    thalch        mean(no_dis)=148.37  mean(disease)=129.07  Δ=-19.30
    oldpeak       mean(no_dis)=0.42  mean(disease)=1.20  Δ=+0.78
    ca            mean(no_dis)=0.11  mean(disease)=0.32  Δ=+0.21
  ✔  Saved → eda_step08_bivariate_numeric_vs_target.png


In [30]:
print("\n" + "═" * 65)
print("  STEP 9 — Bivariate Analysis: Categorical Features vs Target")
print("═" * 65)
 
n_cat2 = len(str_cols)
n_cat2_cols = 3
n_cat2_rows = (n_cat2 + n_cat2_cols - 1) // n_cat2_cols
fig = make_fig(n_cat2_rows, n_cat2_cols, "Step 9 — Bivariate: Categorical Features vs Target (Grouped Bar)", h_per_row=4, w_per_col=7)
gs = gridspec.GridSpec(n_cat2_rows, n_cat2_cols, figure=fig, hspace=0.55, wspace=0.38)
 
for i, col in enumerate(str_cols):
    ax = fig.add_subplot(gs[i // 3, i % 3])
    style_ax(ax, col)
    ct = df.groupby([col, "target"]).size().unstack(fill_value=0)
    ct.columns = ["No Disease", "Disease"]
    x = np.arange(len(ct)); w = 0.38
    ax.bar(x - w/2, ct["No Disease"], w, color=ACC1, label="No Disease", edgecolor="none")
    ax.bar(x + w/2, ct["Disease"],    w, color=ACC2, label="Disease",    edgecolor="none")
    ax.set_xticks(x); ax.set_xticklabels(ct.index, rotation=25, ha="right", fontsize=8)
    ax.set_ylabel("Count")
    ax.legend(frameon=False, labelcolor=TXT, fontsize=8)
    ax.yaxis.grid(True, color=GRID, lw=0.5); ax.set_axisbelow(True)
    print(f"\n    {col}:")
    print(f"    {ct.to_dict()}")
 
savefig(fig, 9, "bivariate_categorical_vs_target")



═════════════════════════════════════════════════════════════════
  STEP 9 — Bivariate Analysis: Categorical Features vs Target
═════════════════════════════════════════════════════════════════

    sex:
    {'No Disease': {'Female': 144, 'Male': 267}, 'Disease': {'Female': 50, 'Male': 459}}

    cp:
    {'No Disease': {'asymptomatic': 104, 'atypical angina': 150, 'non-anginal': 131, 'typical angina': 26}, 'Disease': {'asymptomatic': 392, 'atypical angina': 24, 'non-anginal': 73, 'typical angina': 20}}

    fbs:
    {'No Disease': {False: 367, True: 44}, 'Disease': {False: 415, True: 94}}

    restecg:
    {'No Disease': {'lv hypertrophy': 82, 'normal': 268, 'st-t abnormality': 61}, 'Disease': {'lv hypertrophy': 106, 'normal': 285, 'st-t abnormality': 118}}

    exang:
    {'No Disease': {False: 356, True: 55}, 'Disease': {False: 227, True: 282}}

    slope:
    {'No Disease': {'downsloping': 14, 'flat': 272, 'upsloping': 125}, 'Disease': {'downsloping': 49, 'flat': 382, 'upsloping': 

In [31]:
print("\n" + "═" * 65)
print("  STEP 10 — Correlation Analysis")
print("═" * 65)
 
corr = df_enc.corr()
corr_target = corr["target"].drop("target").sort_values(key=abs, ascending=False)
 
print(f"\n  Top correlations with target:")
for feat, val in corr_target.items():
    bar = "█" * int(abs(val) * 30)
    direction = "+" if val > 0 else "-"
    print(f"    {feat:<12}  {direction}{bar}  {val:+.3f}")
 
fig = make_fig(1, 2, "Step 10 — Correlation Analysis", h_per_row=7, w_per_col=9)
gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)
 
# Full heatmap
ax1 = fig.add_subplot(gs[0])
style_ax(ax1, "Full Correlation Matrix")
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, ax=ax1,
            linewidths=0.4, cbar_kws={"shrink": 0.7},
            annot=True, fmt=".2f", annot_kws={"size": 7, "color": "white"})
ax1.tick_params(axis="x", rotation=45, colors=TXT, labelsize=8)
ax1.tick_params(axis="y", colors=TXT, labelsize=8)
 
# Target correlation bar
ax2 = fig.add_subplot(gs[1])
style_ax(ax2, "Correlation with Target (sorted by |r|)")
bar_c = [ACC2 if v > 0 else ACC1 for v in corr_target.values]
ax2.barh(corr_target.index[::-1], corr_target.values[::-1], color=bar_c[::-1], edgecolor="none", height=0.65)
ax2.axvline(0, color=TXT, lw=0.8, ls="--")
ax2.set_xlabel("Pearson r")
ax2.xaxis.grid(True, color=GRID, lw=0.5); ax2.set_axisbelow(True)
savefig(fig, 10, "correlation_analysis")



═════════════════════════════════════════════════════════════════
  STEP 10 — Correlation Analysis
═════════════════════════════════════════════════════════════════

  Top correlations with target:
    exang         +█████████████  +0.434
    cp            -███████████  -0.385
    thalch        -███████████  -0.382
    oldpeak       +██████████  +0.366
    sex           +█████████  +0.307
    age           +████████  +0.283
    chol          -██████  -0.229
    slope         -██████  -0.205
    thal          +█████  +0.173
    ca            +████  +0.165
    fbs           +███  +0.108
    trestbps      +███  +0.101
    restecg       +█  +0.059
  ✔  Saved → eda_step10_correlation_analysis.png


# OUTLIER DETECTION

In [32]:
print("\n" + "═" * 65)
print("  STEP 11 — Outlier Detection (IQR Method)")
print("═" * 65)
 
fig = make_fig(2, 3, "Step 11 — Outlier Detection (IQR Method)", h_per_row=4, w_per_col=7)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
 
outlier_summary = {}
for i, col in enumerate(num_cols):
    ax = fig.add_subplot(gs[i // 3, i % 3])
    style_ax(ax, col)
    data = df[col].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out = ((data < lower) | (data > upper)).sum()
    outlier_summary[col] = {"Q1": round(Q1, 2), "Q3": round(Q3, 2), "IQR": round(IQR, 2),
                            "Lower": round(lower, 2), "Upper": round(upper, 2), "Outliers": n_out}
    print(f"    {col:<12}  IQR={IQR:.2f}  bounds=[{lower:.2f}, {upper:.2f}]  outliers={n_out}")
 
    bp = ax.boxplot(data, vert=True, patch_artist=True, widths=0.5,
                    medianprops=dict(color=TXT, lw=2),
                    whiskerprops=dict(color=TXT), capprops=dict(color=TXT),
                    flierprops=dict(marker="o", color=ACC2, markersize=4, alpha=0.6))
    bp["boxes"][0].set_facecolor(ACC1); bp["boxes"][0].set_alpha(0.7)
    ax.set_ylabel(col); ax.set_xticks([])
    ax.yaxis.grid(True, color=GRID, lw=0.5); ax.set_axisbelow(True)
    ax.text(0.97, 0.97, f"{n_out} outliers", transform=ax.transAxes,
            ha="right", va="top", color=ACC2, fontsize=9, fontweight="bold")
 
savefig(fig, 11, "outlier_detection")



═════════════════════════════════════════════════════════════════
  STEP 11 — Outlier Detection (IQR Method)
═════════════════════════════════════════════════════════════════
    age           IQR=13.00  bounds=[27.50, 79.50]  outliers=0
    trestbps      IQR=20.00  bounds=[90.00, 170.00]  outliers=28
    chol          IQR=89.25  bounds=[43.88, 400.88]  outliers=185
    thalch        IQR=36.00  bounds=[66.00, 210.00]  outliers=2
    oldpeak       IQR=1.50  bounds=[-2.25, 3.75]  outliers=16
    ca            IQR=0.00  bounds=[0.00, 0.00]  outliers=128
  ✔  Saved → eda_step11_outlier_detection.png


# FEATURE INTERACTION ANALYSIS

In [33]:
print("\n" + "═" * 65)
print("  STEP 12 — Feature Interaction Analysis")
print("═" * 65)
 
fig = make_fig(2, 3, "Step 12 — Feature Interactions (Scatter Plots)", h_per_row=5, w_per_col=7)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.48, wspace=0.35)
 
interactions = [
    ("age",      "thalch",   "Age vs Max Heart Rate"),
    ("age",      "oldpeak",  "Age vs ST Depression"),
    ("chol",     "trestbps", "Cholesterol vs Resting BP"),
    ("thalch",   "oldpeak",  "Max HR vs ST Depression"),
    ("age",      "chol",     "Age vs Cholesterol"),
    ("trestbps", "thalch",   "Resting BP vs Max HR"),
]
 
for i, (x_col, y_col, title) in enumerate(interactions):
    ax = fig.add_subplot(gs[i // 3, i % 3])
    style_ax(ax, title)
    for tgt, col_s, lbl in [(0, ACC1, "No Disease"), (1, ACC2, "Disease")]:
        d = df[df["target"] == tgt]
        ax.scatter(d[x_col], d[y_col], color=col_s, alpha=0.35, s=15,
                   label=lbl, edgecolors="none")
    ax.set_xlabel(x_col); ax.set_ylabel(y_col)
    ax.legend(frameon=False, labelcolor=TXT, fontsize=8)
    ax.yaxis.grid(True, color=GRID, lw=0.5); ax.set_axisbelow(True)
    r = df[[x_col, y_col]].corr().iloc[0, 1]
    ax.text(0.97, 0.97, f"r={r:.3f}", transform=ax.transAxes,
            ha="right", va="top", color=ACC4, fontsize=8.5, fontweight="bold")
    print(f"    {x_col} × {y_col:<12}  Pearson r={r:.3f}")
 
savefig(fig, 12, "feature_interactions")



═════════════════════════════════════════════════════════════════
  STEP 12 — Feature Interaction Analysis
═════════════════════════════════════════════════════════════════
    age × thalch        Pearson r=-0.350
    age × oldpeak       Pearson r=0.234
    chol × trestbps      Pearson r=0.089
    thalch × oldpeak       Pearson r=-0.149
    age × chol          Pearson r=-0.086
    trestbps × thalch        Pearson r=-0.105
  ✔  Saved → eda_step12_feature_interactions.png


In [34]:
print("\n" + "═" * 65)
print("  STEP 13 — Statistical Summary by Group")
print("═" * 65)
 
group_stats = df.groupby("target")[num_cols].agg(["mean", "median", "std"]).round(2)
print(f"\n  Group statistics (target=0 vs 1):")
print(group_stats.to_string())
 
fig = make_fig(1, 1, "Step 13 — Mean Feature Values: Disease vs No Disease (Normalised)",
               h_per_row=6, w_per_col=16)
ax = fig.add_subplot(111)
style_ax(ax, "Normalised Mean Feature Values by Target")
means_0 = df[df["target"] == 0][num_cols].mean()
means_1 = df[df["target"] == 1][num_cols].mean()
norm0 = (means_0 - df[num_cols].min()) / (df[num_cols].max() - df[num_cols].min())
norm1 = (means_1 - df[num_cols].min()) / (df[num_cols].max() - df[num_cols].min())
x = np.arange(len(num_cols)); w = 0.38
bars0 = ax.bar(x - w/2, norm0, w, color=ACC1, label="No Disease", edgecolor="none")
bars1 = ax.bar(x + w/2, norm1, w, color=ACC2, label="Disease",    edgecolor="none")
ax.set_xticks(x); ax.set_xticklabels(num_cols, fontsize=10)
ax.set_ylabel("Normalised Mean (0–1)")
ax.legend(frameon=False, labelcolor=TXT, fontsize=10)
ax.yaxis.grid(True, color=GRID, lw=0.5); ax.set_axisbelow(True)
savefig(fig, 13, "group_summary")



═════════════════════════════════════════════════════════════════
  STEP 13 — Statistical Summary by Group
═════════════════════════════════════════════════════════════════

  Group statistics (target=0 vs 1):
          age              trestbps                  chol                 thalch               oldpeak                 ca             
         mean median   std     mean median    std    mean median     std    mean median    std    mean median   std  mean median   std
target                                                                                                                                
0       50.55   51.0  9.43   129.92  130.0  16.45  227.68  225.0   74.06  148.37  150.0  23.10    0.42    0.0  0.70  0.11    0.0  0.43
1       55.90   57.0  8.72   133.67  130.0  19.78  177.49  219.0  126.31  129.07  130.0  23.37    1.20    1.0  1.17  0.32    0.0  0.74
  ✔  Saved → eda_step13_group_summary.png


# KEY FINDINGS & INSIGHTS

In [35]:
print("\n" + "═" * 65)
print("  STEP 14 — Key Findings & Insights")
print("═" * 65)
 
insights = [
    ("Target Balance",      "Dataset is reasonably balanced: 44.7% no-disease vs 55.3% disease."),
    ("Age",                 "Disease patients are slightly older (avg ~56) vs healthy (avg ~52)."),
    ("Sex",                 "Males are at significantly higher risk than females in this dataset."),
    ("Max Heart Rate",      "Disease patients have notably lower max HR (avg ~140 vs ~158). Strong predictor."),
    ("ST Depression",       "Oldpeak is higher in disease patients (avg ~1.6 vs ~0.6). Key feature."),
    ("Chest Pain",          "Asymptomatic CP is most common in disease patients — counterintuitive but crucial."),
    ("Thalassemia",         "Reversible defect is a strong indicator of heart disease presence."),
    ("Vessels (ca)",        "More blocked vessels → higher disease probability. Near-zero ca = healthier."),
    ("Cholesterol",         "Weak individual predictor; similar median across groups."),
    ("Blood Pressure",      "Slight increase in disease group but overlap is high; weak alone."),
    ("Best Correlators",    "thalch (−), ca (+), oldpeak (+), cp (+) are strongest vs target."),
]
 
fig = make_fig(1, 1, "Step 14 — Key EDA Findings & Clinical Insights",
               h_per_row=8, w_per_col=18)
ax = fig.add_subplot(111)
ax.set_facecolor(CARD)
ax.axis("off")
 
ax.text(0.5, 0.98, "KEY FINDINGS FROM EXPLORATORY DATA ANALYSIS",
        transform=ax.transAxes, ha="center", va="top",
        fontsize=14, color=ACC4, fontweight="bold")
 
for i, (topic, insight) in enumerate(insights):
    y_pos = 0.90 - i * 0.075
    ax.text(0.02, y_pos, f"▸ {topic}",
            transform=ax.transAxes, va="top", ha="left",
            fontsize=11, color=ACC1, fontweight="bold")
    ax.text(0.22, y_pos, insight,
            transform=ax.transAxes, va="top", ha="left",
            fontsize=10.5, color=TXT, wrap=True)
    ax.plot([0.01, 0.99], [y_pos - 0.055, y_pos - 0.055],
            color=GRID, lw=0.6, transform=ax.transAxes, clip_on=False)
 
print(f"\n  ┌{'─'*60}┐")
for topic, insight in insights:
    print(f"  │  ▸ {topic:<22}  {insight[:46]:<46}│")
print(f"  └{'─'*60}┘")
savefig(fig, 14, "key_findings")



═════════════════════════════════════════════════════════════════
  STEP 14 — Key Findings & Insights
═════════════════════════════════════════════════════════════════

  ┌────────────────────────────────────────────────────────────┐
  │  ▸ Target Balance          Dataset is reasonably balanced: 44.7% no-disea│
  │  ▸ Age                     Disease patients are slightly older (avg ~56) │
  │  ▸ Sex                     Males are at significantly higher risk than fe│
  │  ▸ Max Heart Rate          Disease patients have notably lower max HR (av│
  │  ▸ ST Depression           Oldpeak is higher in disease patients (avg ~1.│
  │  ▸ Chest Pain              Asymptomatic CP is most common in disease pati│
  │  ▸ Thalassemia             Reversible defect is a strong indicator of hea│
  │  ▸ Vessels (ca)            More blocked vessels → higher disease probabil│
  │  ▸ Cholesterol             Weak individual predictor; similar median acro│
  │  ▸ Blood Pressure          Slight increase in dise

In [36]:
print("\n" + "═" * 65)
print("EDA COMPLETE — 14 steps finished!")
print("PNG charts saved as:  eda_step01_... to eda_step14_...")
print("=" * 65 + "\n")


═════════════════════════════════════════════════════════════════
EDA COMPLETE — 14 steps finished!
PNG charts saved as:  eda_step01_... to eda_step14_...

